In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies = pd.read_csv("BollywoodMovieDetail.csv")

In [ ]:
movies.head()

,imdbId,title,releaseYear,releaseDate,genre,writers,actors,directors,sequel,hitFlop
0,tt0118578,Albela,2001,20 Apr 2001,Romance,Honey Irani (screenplay) | Honey Irani (story)...,Govinda | Aishwarya Rai Bachchan | Jackie Shro...,Deepak Sareen,0.0,2
1,tt0169102,Lagaan: Once Upon a Time in India,2001,08 May 2002,Adventure | Drama | Musical,Ashutosh Gowariker (story) | Ashutosh Gowarike...,Aamir Khan | Gracy Singh | Rachel Shelley | Pa...,Ashutosh Gowariker,0.0,6
2,tt0187279,Meri Biwi Ka Jawab Nahin,2004,02 Jul 2004,Action | Comedy,NaN,Akshay Kumar | Sridevi | Gulshan Grover | Laxm...,Pankaj Parashar | S.M. Iqbal,0.0,1
3,tt0222024,Hum Tumhare Hain Sanam,2002,24 May 2002,Drama | Romance,K.S. Adiyaman | Arun Kumar (assistant dialogue...,Shah Rukh Khan | Madhuri Dixit | Salman Khan |...,K.S. Adiyaman,0.0,4
4,tt0227194,One 2 Ka 4,2001,30 Mar 2001,Action | Comedy | Drama,Sanjay Chhel | Raaj Kumar Dahima (screenplay) ...,Shah Rukh Khan | Juhi Chawla | Jackie Shroff |...,Shashilal K. Nair,0.0,1


In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1284 entries, 0 to 1283
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   imdbId       1284 non-null   object 
 1   title        1284 non-null   object 
 2   releaseYear  1284 non-null   int64  
 3   releaseDate  1231 non-null   object 
 4   genre        1282 non-null   object 
 5   writers      1165 non-null   object 
 6   actors       1281 non-null   object 
 7   directors    1280 non-null   object 
 8   sequel       1281 non-null   float64
 9   hitFlop      1284 non-null   int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 100.4+ KB


In [ ]:
movies.isnull().sum()

,0
imdbId,0
title,0
releaseYear,0
releaseDate,53
genre,2
writers,119
actors,3
directors,4
sequel,3
hitFlop,0


In [ ]:
movies = movies[
[
    "title",
    "genre",
    "actors",
    "directors",
    "writers",
    "releaseYear",
    "hitFlop",
    "sequel"
]
]

In [ ]:
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

In [ ]:
def clean(text):
    text = str(text)
    text = text.replace("|", ", ")
    text = text.replace("(", "")
    text = text.replace(")", "")
    return text.strip()

In [ ]:
movies["genre"] = movies["genre"].apply(clean)
movies["actors"] = movies["actors"].apply(clean)
movies["directors"] = movies["directors"].apply(clean)
movies["writers"] = movies["writers"].apply(clean)

In [ ]:
movies["tags"] = (

movies["genre"]+" "+
movies["genre"]+" "+
movies["genre"]+" "+

movies["actors"]+" "+
movies["actors"]+" "+

movies["directors"]+" "+
movies["directors"]+" "+
movies["directors"]+" "+

movies["writers"]
)

In [ ]:
movies["tags"] = movies["tags"].str.lower()

In [ ]:
new_df = movies[
[
"title",
"genre",
"actors",
"directors",
"releaseYear",
"hitFlop",
"sequel",
"tags"
]
]

In [ ]:
cv = CountVectorizer(
max_features=5000,
stop_words="english"
)

In [ ]:
vectors = cv.fit_transform(
new_df["tags"]
).toarray()

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
pickle.dump(
new_df,
open("movies.pkl","wb")
)

In [ ]:
pickle.dump(
similarity,
open("similarity.pkl","wb")
)

In [ ]:
indices = pd.Series(
new_df.index,
index=new_df["title"]
).drop_duplicates()

In [ ]:
def recommend(movie):

    if movie not in indices.index:
        return "Movie not found!!"

    idx = indices[movie]

    distances = list(
        enumerate(
            similarity[idx]
        )
    )

    distances = sorted(
        distances,
        key=lambda x:x[1],
        reverse=True
    )

    recommended=[]

    for i in distances[1:6]:

        row = new_df.iloc[i[0]]

        recommended.append({

            "title":row["title"],

            "genre":row["genre"],

            "actors":row["actors"],

            "director":row["directors"],

            "year":row["releaseYear"],

            "hitFlop":row["hitFlop"]
        })
    return recommended

In [ ]:
movies["actors"].head()

,actors
0,"Govinda , Aishwarya Rai Bachchan , Jackie Sh..."
1,"Aamir Khan , Gracy Singh , Rachel Shelley , ..."
2,"Shah Rukh Khan , Madhuri Dixit , Salman Khan..."
3,"Shah Rukh Khan , Juhi Chawla , Jackie Shroff..."
4,"Shah Rukh Khan , Madhuri Dixit , Aishwarya R..."
